In [1]:
!uv add numpy
!uv add pandas
!uv add nltk
!uv add contractions
!uv add scikit-learn
!uv add tqdm
!uv add joblib
!uv add tensorflow
!uv add keras

Resolved 58 packages in 13ms
Checked 55 packages in 9ms
Resolved 58 packages in 5ms
Checked 55 packages in 3ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms
Resolved 58 packages in 4ms
Checked 55 packages in 2ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms
Resolved 58 packages in 5ms
Checked 55 packages in 3ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms
Resolved 58 packages in 4ms
Checked 55 packages in 3ms


In [2]:
import pandas as pd
import numpy as np
import contractions
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight
import re
import os
import joblib

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# download words
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)

True

In [5]:
os.makedirs('models', exist_ok=True)
os.makedirs('output', exist_ok=True)

In [6]:
df = pd.read_csv('./data/equal.csv', encoding='latin-1')

df['Sentiment'] = df['Sentiment'].str.lower().str.strip()
df = df.dropna(subset=['Review', 'Sentiment'])
df = df.reset_index(drop=True)

df.head(3)

,product_name,product_price,Rate,Review,Summary,Sentiment
0,SportSoul Cotton Gym & Athletic Abdomen Suppor...,449,4,good quality product,good cuality,positive
1,SportSoul Cotton Gym & Athletic Abdomen Suppor...,449,4,wonderful,super product,positive
2,SportSoul Cotton Gym & Athletic Abdomen Suppor...,449,4,value-for-money,as expected,positive


In [7]:
# preprocess with util functions
lemmatizer = WordNetLemmatizer()

STOPWORDS_SET = set(stopwords.words('english'))
NEGATION_WORDS = {'not', 'no', 'never', "n't", 'neither', 'nor', 'none'}

# don't remove negation words since it shows emotion
STOPWORDS_SET -= NEGATION_WORDS

def expand_contractions(text):
    try:
        return contractions.fix(text)
    except Exception:
        return text

def remove_html_tags(text):
    return re.sub(r'<[^>]+>', ' ', text)

def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)


def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text)


def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()


def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in STOPWORDS_SET]
    return ' '.join(filtered)


def lemmatize_text(text):
    words = text.split()
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized)


def handle_emojis(text):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)


In [8]:
# utility function to full preprocess
def full_preprocess(text, use_lemmatization=True):
    if not isinstance(text, str):
        return ""

    text = expand_contractions(text)
    text = text.lower()
    text = remove_html_tags(text)
    text = remove_urls(text)
    text = handle_emojis(text)
    text = remove_special_characters(text)
    text = remove_stopwords(text)

    if use_lemmatization:
        text = lemmatize_text(text)

    text = remove_extra_spaces(text)

    return text


In [9]:
# make the combined text cols
df['combined_text'] = (
    df['Summary'].fillna('') + ' ' + df['Review'].fillna('')
)
df.head(1)

,product_name,product_price,Rate,Review,Summary,Sentiment,combined_text
0,SportSoul Cotton Gym & Athletic Abdomen Suppor...,449,4,good quality product,good cuality,positive,good cuality good quality product


In [10]:
from tqdm import tqdm
tqdm.pandas(desc="Preprocessing")

df['clean_text'] = df['combined_text'].progress_apply(
    lambda x: full_preprocess(x, use_lemmatization=True)
)

df = df[df['clean_text'].str.len() > 5]
df = df.reset_index(drop=True)
df.head(1)

Preprocessing: 100%|██████████| 76704/76704 [00:02<00:00, 37950.09it/s]


,product_name,product_price,Rate,Review,Summary,Sentiment,combined_text,clean_text
0,SportSoul Cotton Gym & Athletic Abdomen Suppor...,449,4,good quality product,good cuality,positive,good cuality good quality product,good cuality good quality product


In [11]:
le = LabelEncoder()

# fit_transform() first "learns" the unique labels, then transforms them
df['label'] = le.fit_transform(df['Sentiment'])

print("   Label encoding map:")
for i, cls in enumerate(le.classes_):
    print(f"     {cls} → {i}")

joblib.dump(le, './models/label_encoder.pkl')

   Label encoding map:
     negative → 0
     neutral → 1
     positive → 2


['./models/label_encoder.pkl']

In [12]:
# spliting
X = df['clean_text'].values    # Features (the cleaned text)
y = df['label'].values          # Labels (0, 1, or 2)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"   Train:      {len(X_train):,} samples")
print(f"   Validation: {len(X_val):,} samples")
print(f"   Test:       {len(X_test):,} samples")

   Train:      53,670 samples
   Validation: 11,501 samples
   Test:       11,501 samples


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 30000
MAX_LEN = 100

# Create tokenizer
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token='<OOV>',
    lower=True
)

tokenizer.fit_on_texts(X_train)

print(f"   Vocabulary size: {len(tokenizer.word_index):,} unique words")

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f"   Train sequences shape: {X_train_pad.shape}")

# serialize tokenizer and pad_sequences for reproducibility
joblib.dump(tokenizer, 'models/tokenizer.pkl')

# class_weight.compute_class_weight() calculates balanced weights
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(weights))
print(f"   Class weights: {class_weights_dict}")

np.save('models/X_train_pad.npy', X_train_pad)
np.save('models/X_val_pad.npy',   X_val_pad)
np.save('models/X_test_pad.npy',  X_test_pad)
np.save('models/y_train.npy',     y_train)
np.save('models/y_val.npy',       y_val)
np.save('models/y_test.npy',      y_test)
np.save('models/class_weights.npy', weights)

np.save('models/X_train_raw.npy', X_train)
np.save('models/X_val_raw.npy',   X_val)
np.save('models/X_test_raw.npy',  X_test)

joblib.dump({'MAX_WORDS': MAX_WORDS, 'MAX_LEN': MAX_LEN}, 'models/config.pkl')

   Vocabulary size: 18,263 unique words
   Train sequences shape: (53670, 100)
   Class weights: {0: np.float64(1.0222273012970688), 1: np.float64(0.9922351636161952), 2: np.float64(0.9862726721428965)}


['models/config.pkl']